[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/15_Export.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 导出模块（Export Module）—— 全面指南

## 概览

本 notebook 全面介绍 Semantica 的**导出模块**：把知识图谱、实体、关系和数据导出成多种格式。模块支持 **8 种导出格式**（RDF、JSON、CSV、Graph、YAML、OWL、Vector、LPG），外加报告生成能力。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/export/)

### 先看主线

前面攒下的家底——07–10 篇的图谱、12/13 篇的向量、14 篇的本体——在这一篇各找各的出口。导出是流水线的最后一公里，关键是**按目的地选格式**：

**通用交换（第 1–2 步）**

JSON、CSV——最通用的两种数据格式，给脚本、pandas、任何程序。

**语义网（第 3、5 步）**

RDF 五种写法 + OWL 本体——交给语义网世界，还自带校验。

**专用工具（第 4、6 步）**

Graph 格式给可视化工具（Gephi、Cytoscape、Graphviz）、Vector 给向量库、LPG 给图数据库（Neo4j/Memgraph）、报告给人看。

预告一个坑：第 5 步的 `OWLExporter` 要的是**本体结构**而不是知识图谱——"楼"和"法"不能混着交（14 篇的比方），先用 `OntologyGenerator` 转换。代码里就是这么做的。

### 导出模块架构

导出模块由以下部分组成：

#### **核心导出类**（9 个）
- `RDFExporter` - RDF 格式导出（Turtle、RDF/XML、JSON-LD、N-Triples、N3）
- `JSONExporter` - JSON 与 JSON-LD 格式导出
- `CSVExporter` - 面向表格数据的 CSV 导出
- `GraphExporter` - 图格式导出（GraphML、GEXF、DOT）
- `SemanticNetworkYAMLExporter` - 语义网络 YAML 导出
- `YAMLSchemaExporter` - schema YAML 导出
- `OWLExporter` - 面向本体的 OWL 格式导出
- `VectorExporter` - 面向向量库的向量嵌入导出
- `LPGExporter` - 面向 Neo4j、Memgraph 等数据库的 LPG 格式导出

#### **支撑类**（4 个）
- `RDFSerializer` - 格式互转的 RDF 序列化引擎
- `RDFValidator` - 语法检查的 RDF 校验引擎
- `NamespaceManager` - RDF 命名空间管理与冲突消解
- `ReportGenerator` - 报告生成（HTML、Markdown、JSON、Text）

#### **注册与配置**（2 类 + 2 实例）
- `MethodRegistry` - 自定义导出方法的注册表
- `method_registry` - 全局注册表实例
- `ExportConfig` - 导出模块的配置管理器
- `export_config` - 全局配置实例

### 学习目标

完成本 notebook 后，你将能够：

- 把知识图谱导出为全部支持格式（RDF、JSON、CSV、Graph、YAML、OWL、Vector、LPG）
- 直接使用导出器类做精细控制
- 以多种格式生成专业报告
- 注册并使用自定义导出方法
- 通过环境变量或配置文件配置导出设置
- 使用 RDF 序列化、校验和命名空间管理

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

## 第 1 步：JSON 导出

通用交换第一站。用 `JSONExporter` 把知识图谱导出成 JSON。

**JSONExporter 方法：**
- `export()` - 把任意数据导出为 JSON
- `export_knowledge_graph()` - 导出知识图谱为 JSON/JSON-LD
- `export_entities()` - 导出实体为 JSON
- `export_relationships()` - 导出关系为 JSON

In [ ]:
!pip install semantica

In [ ]:
from semantica.export import JSONExporter
from semantica.kg import GraphBuilder

# Create exporter and builder
json_exporter = JSONExporter()
builder = GraphBuilder()

# Create sample knowledge graph
entities = [{"id": "e1", "type": "Organization", "name": "Apple Inc.", "properties": {}}]
relationships = []

kg = builder.build(entities + relationships)

# Export to JSON
json_exporter.export_knowledge_graph(kg, "output.json")


## 第 2 步：CSV 导出

表格世界的通行证。用 `CSVExporter` 把实体和关系导出成 CSV。

**CSVExporter 方法：**
- `export()` - 导出知识图谱为 CSV
- `export_entities()` - 导出实体到 CSV 文件
- `export_relationships()` - 导出关系到 CSV 文件
- `export_knowledge_graph()` - 导出完整知识图谱到 CSV

In [ ]:
from semantica.export import CSVExporter

# Create CSV exporter
csv_exporter = CSVExporter()

# Export entities to CSV
csv_exporter.export_entities(entities, "entities.csv")


## 第 3 步：RDF 导出

进入语义网世界。用 `RDFExporter` 导出 RDF——一种数据，五种写法：

**RDFExporter 方法：**
- `export()` - 导出为 RDF（支持多种格式）
- `export_knowledge_graph()` - 导出知识图谱为 RDF
- `export_entities()` - 导出实体为 RDF
- `export_relationships()` - 导出关系为 RDF

**支持的 RDF 格式：**
- `turtle` - Turtle 格式（人类可读）
- `rdfxml` - RDF/XML 格式
- `jsonld` - JSON-LD 格式
- `ntriples` - N-Triples 格式
- `n3` - N3 格式

**其他 RDF 类：**
- `RDFSerializer` - 在 RDF 格式之间互转
- `RDFValidator` - 校验 RDF 语法与一致性
- `NamespaceManager` - 管理 RDF 命名空间

In [ ]:
from semantica.export import RDFExporter

# Create RDF exporter
rdf_exporter = RDFExporter()

# Export to RDF format (Turtle by default)
rdf_exporter.export(kg, "output.ttl", format="turtle")

In [ ]:
# TTL alias: format="ttl" is equivalent to format="turtle"
rdf_data = {
    "entities": [
        {"id": "e1", "text": "Apple Inc.", "type": "ORG", "confidence": 0.95},
        {"id": "e2", "text": "Steve Jobs", "type": "PERSON", "confidence": 0.97},
    ],
    "relationships": [
        {"source_id": "e2", "target_id": "e1", "type": "founded_by", "confidence": 0.91},
    ],
}

rdf_exporter.export(rdf_data, "output.ttl", format="ttl")

result = rdf_exporter.validate_rdf(rdf_data)
print(f"Valid: {result['overall_valid']}")

## 第 4 步：图格式导出

给可视化工具备料。用 `GraphExporter` 导出图格式——每种格式对应一类工具：

**GraphExporter 方法：**
- `export()` - 导出为图格式
- `export_knowledge_graph()` - 导出知识图谱为图格式

**支持的图格式：**
- `graphml` - GraphML 格式（给 Cytoscape、yEd 等）
- `gexf` - GEXF 格式（给 Gephi）
- `dot` - Graphviz DOT 格式

In [ ]:
from semantica.export import GraphExporter

# Create graph exporter
graph_exporter = GraphExporter()

# Export to GraphML format
graph_exporter.export_knowledge_graph(kg, "output.graphml", format="graphml")


## 第 5 步：OWL 导出

14 篇的本体在这里落地。用 `OWLExporter` 把本体导出为 OWL 格式——注意预告过的坑，它要本体不要图谱。

**OWLExporter 方法：**
- `export()` - 导出本体为 OWL
- `export_ontology()` - 导出完整本体
- `export_classes()` - 只导出类定义
- `export_properties()` - 只导出属性定义

**支持的 OWL 格式：**
- `owl-xml` - OWL/XML 格式（默认）
- `turtle` - Turtle 格式的 OWL

**注意：** `OWLExporter` 期望的是本体结构（含 classes、properties），不是知识图谱。先用 `OntologyGenerator` 把知识图谱转换成本体。

In [ ]:
from semantica.export import OWLExporter
from semantica.ontology import OntologyGenerator

# Create OWL exporter and ontology generator
owl_exporter = OWLExporter()
generator = OntologyGenerator()

# Generate ontology from entities and relationships
# Note: Pass data as a dictionary to generate_ontology
ontology = generator.generate_ontology({
    "entities": entities,
    "relationships": relationships
})

# Export ontology to OWL
owl_exporter.export(ontology, "output.owl")

## 第 6 步：更多导出格式

长尾格式各有其主：

### YAML 导出

用 YAML 导出器导出为 YAML 格式：

**YAML 导出器类：**
- `SemanticNetworkYAMLExporter` - 语义网络导出为 YAML
- `YAMLSchemaExporter` - 本体 schema 导出为 YAML

### 向量导出

用 `VectorExporter` 导出向量嵌入——12/13 篇的向量交给别的向量库：

**VectorExporter 方法：**
- `export()` - 导出向量为多种格式

**支持的向量格式：**
- `json` - JSON 格式
- `numpy` - NumPy 格式
- `binary` - 二进制格式
- `faiss` - FAISS 格式

### LPG 导出

用 `LPGExporter` 导出为带标签属性图（LPG）——09 篇图数据库的专线：

**LPGExporter 方法：**
- `export()` - 导出为 LPG 格式
- `export_knowledge_graph()` - 导出知识图谱为 LPG

**支持的 LPG 格式：**
- `cypher` - Cypher 查询格式（给 Neo4j、Memgraph）
- `lpg` - 带标签属性图格式

### 报告生成

用 `ReportGenerator` 生成专业报告——给人看的出口：

**ReportGenerator 方法：**
- `generate_report()` - 以多种格式生成报告

**支持的报告格式：**
- `html` - HTML 报告
- `markdown` - Markdown 报告
- `json` - JSON 报告
- `text` - 纯文本报告


## 小结

### 回顾主线

最后一公里走完：**通用交换**（JSON、CSV）谁都能接；**语义网**（RDF 五连、OWL）进入标准生态；**专用工具**（Graph 给可视化、Vector 给向量库、LPG 给图数据库、报告给人）各走各的专线。家底再多，出得去才算数。

### 你已学会

**核心导出类：**
- **JSONExporter**：导出为 JSON/JSON-LD 格式
- **CSVExporter**：导出为 CSV 格式
- **RDFExporter**：导出为 RDF 格式（Turtle、RDF/XML、JSON-LD、N-Triples、N3）
- **GraphExporter**：导出为图格式（GraphML、GEXF、DOT）
- **OWLExporter**：导出本体为 OWL
- **VectorExporter**：导出向量为多种格式
- **LPGExporter**：导出为带标签属性图格式
- **SemanticNetworkYAMLExporter**：语义网络导出为 YAML
- **YAMLSchemaExporter**：schema 导出为 YAML

**支撑类：**
- **RDFSerializer**：RDF 格式互转
- **RDFValidator**：RDF 校验
- **NamespaceManager**：RDF 命名空间管理
- **ReportGenerator**：生成专业报告

**注册与配置：**
- `MethodRegistry`：注册自定义导出方法
- `ExportConfig`：配置导出设置
- `method_registry`：访问已注册方法的全局注册表实例
- `export_config`：导出设置的全局配置实例

**下一步：**
- 在[多格式导出 notebook](../advanced/05_Multi_Format_Export.ipynb)学习高级导出技巧
- 在 Visualization notebook 学习数据可视化